# CSE ETL data-quality validation

This notebook is the reproducible release gate for extraction coverage, certainty, duplicate keys, unit handling, historical prices, and the manually verified golden fixture set. Run it from the project root after `cse-etl run`.

In [ ]:
from datetime import date
from pathlib import Path
import json
import polars as pl
from cse_financial_etl.validation.golden import validate_golden

PROJECT_ROOT = Path.cwd()
FACTS_PATH = PROJECT_ROOT / 'data/gold/current_financial_facts.parquet'
PRICES_PATH = PROJECT_ROOT / 'data/gold/current_market_prices.parquet'
MANIFEST_PATH = PROJECT_ROOT / 'outputs/run_manifest_2026-09-03.json'
facts = pl.read_parquet(FACTS_PATH)
prices = pl.read_parquet(PRICES_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
facts.shape, prices.shape

In [ ]:
status_summary = (facts.group_by('status').len().sort('len', descending=True)
                  .with_columns((pl.col('len') / pl.col('len').sum()).alias('rate')))
status_summary

In [ ]:
accepted = ['EXTRACTED', 'EXTRACTED_DERIVED']
coverage = (facts.group_by('metric_code').agg(
    pl.len().alias('records'),
    pl.col('status').is_in(accepted).sum().alias('accepted'),
    pl.col('overall_certainty').mean().alias('mean_certainty'),
).with_columns((pl.col('accepted') / pl.col('records')).alias('coverage_rate'))
.sort('coverage_rate'))
coverage

In [ ]:
price_summary = prices.group_by('status').len().sort('len', descending=True)
price_summary

In [ ]:
duplicate_facts = (facts.group_by(['issuer_name', 'period_end', 'metric_code'])
                   .len().filter(pl.col('len') > 1))
duplicate_prices = (prices.group_by(['symbol', 'period_end'])
                    .len().filter(pl.col('len') > 1))
{'duplicate_fact_keys': duplicate_facts.height, 'duplicate_price_keys': duplicate_prices.height}

In [ ]:
golden = validate_golden(PROJECT_ROOT, date.fromisoformat(manifest['as_of_date']))
{'sample_size': golden['sample_size'], 'accuracy': golden['accuracy']}

In [ ]:
status_counts = dict(zip(status_summary['status'].to_list(), status_summary['len'].to_list()))
release_gates = {
    'pipeline_errors_zero': manifest['pipeline_error_count'] == 0,
    'golden_accuracy_100pct': golden['accuracy'] == 1.0,
    'golden_sample_at_least_40': golden['sample_size'] >= 40,
    'duplicate_fact_keys_zero': duplicate_facts.height == 0,
    'duplicate_price_keys_zero': duplicate_prices.height == 0,
}
coverage_targets = {
    'unit_not_detected_zero': status_counts.get('UNIT_NOT_DETECTED', 0) == 0,
    'exact_quarter_exception_zero': status_counts.get('EXACT_QUARTER_NOT_REPORTED', 0) == 0,
    'historical_price_exception_zero': prices.filter(pl.col('status') == 'HISTORICAL_PRICE_NOT_AVAILABLE').height == 0,
}
{'release_gates': release_gates, 'coverage_targets': coverage_targets}

In [ ]:
assert all(release_gates.values()), {key: value for key, value in release_gates.items() if not value}
unmet_coverage = [key for key, value in coverage_targets.items() if not value]
print('PASS: structural and measured-accuracy release gates are satisfied')
print('Coverage remediation still required:', unmet_coverage or 'none')